# Imports

In [ ]:
import sys
import os
import numpy as np
import pickle

sys.path.append(os.path.abspath('../src'))
from D_Compute_Spectrograms.load_eeg_epochs import load_eeg_epochs
from C_EEG_Epoching.load_clean_eeg import load_clean_eeg
from D_Compute_Spectrograms.load_interpolation_config import load_interpolation_config
from D_Compute_Spectrograms.interpolate_epoch_groups import interpolate_epoch_groups
from D_Compute_Spectrograms.create_epochs_from_interpolated_data import create_epochs_from_interpolated_data
from D_Compute_Spectrograms.compute_tfr_by_event import compute_tfr_by_event
from E_Plot_Spectrograms.average_grouped_tfr import average_grouped_tfr
from D_Compute_Spectrograms.save_tfr_results import save_tfr_results


# Variables

In [ ]:
subjectID = 'Pilote002'
date      = '2026-07-10'
task      = 'Task1_PPS'
data_path =  '/Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Data'
file_name = f"{subjectID}_{date}_{task}"

baseline_time_start = -0.7
baseline_time_stop = 0.0
padding_time = 0.2
N_baseline_points = 900 # Contain the padding period
fmin = 4
fmax = 40

### Load data

In [ ]:
file_path = os.path.join(data_path, subjectID, task,'Output','epoched_EEG')
file_name = f"{subjectID}_{date}_{task}"
epochs, events_id, events_detailed = load_eeg_epochs(file_path, file_name)

raw_cleaned, sfreq, descriptions = load_clean_eeg(data_path, subjectID, date, task)

config_path_raw = os.path.join(os.path.dirname(data_path), 'Code', 'EEG_analysis', 'Scripts', 'src', 'D_Compute_Spectrograms', 'interpolation_config_per_trial_type.json')
interpolation_config_raw = load_interpolation_config(config_path_raw)

config_path_marged = os.path.join(os.path.dirname(data_path), 'Code', 'EEG_analysis', 'Scripts', 'src', 'D_Compute_Spectrograms', 'interpolation_config_for_merging.json')
interpolation_config_merged = load_interpolation_config(config_path_marged)

### Interpolate raw Epochs

In [ ]:
if task=='Task1_PPS':
    interp_data_raw, _ = interpolate_epoch_groups(
        epochs,
        events_detailed,
        raw_cleaned,
        N_baseline_points,
        interpolation_config_raw
    )

    sfreq = raw_cleaned.info["sfreq"]
    interpolated_epochs_raw, _ = create_epochs_from_interpolated_data(interp_data_raw, sfreq, N_baseline_points, baseline_time_start-padding_time, ch_names=epochs.ch_names)

### Interpolate for merging

In [ ]:
if task=='Task1_PPS':
    interp_data_merged, _ = interpolate_epoch_groups(
        epochs,
        events_detailed,
        raw_cleaned,
        N_baseline_points,
        interpolation_config_merged
    )

    sfreq = raw_cleaned.info["sfreq"]
    interpolated_epochs_merged, _ = create_epochs_from_interpolated_data(interp_data_merged, sfreq, N_baseline_points, baseline_time_start-padding_time, ch_names=epochs.ch_names)

### Wavelet computation

In [ ]:
freqs = np.arange(fmin, fmax, 1)

if task=='Task1_PPS':
    tfr_single_raw, tfr_average_raw = compute_tfr_by_event(
        interpolated_epochs_raw,
        freqs=freqs,
        baseline=(baseline_time_start, baseline_time_stop),
        baseline_mode="logratio"
    )
    tfr_single_merged, tfr_average_merged = compute_tfr_by_event(
        interpolated_epochs_merged,
        freqs=freqs,
        baseline=(baseline_time_start, baseline_time_stop),
        baseline_mode="logratio"
    )
    
elif task=='Task2_HitOrMiss':
    tfr_single_raw, tfr_average_raw = compute_tfr_by_event(
            epochs,
            freqs=freqs,
            baseline=(baseline_time_start, baseline_time_stop),
            baseline_mode="logratio"
        )

### Grouped per trials type

In [ ]:
if task=='Task1_PPS':
    groups = {
        "Both_Near_Fast": ["Both_D1_Narrow_Fast", "Both_D2_Narrow_Fast", "Both_D3_Narrow_Fast"],
        "Both_Near_Slow": ["Both_D1_Narrow_Slow", "Both_D2_Narrow_Slow", "Both_D3_Narrow_Slow"],

        "Both_Far_Fast": ["Both_D6_Narrow_Fast", "Both_D5_Narrow_Fast", "Both_D4_Narrow_Fast"],
        "Both_Far_Slow": ["Both_D6_Narrow_Slow", "Both_D5_Narrow_Slow", "Both_D4_Narrow_Slow"],

        "VibrotactileOnly_Near_Fast": ["VibrotactileOnly_D1_Narrow_Fast", "VibrotactileOnly_D2_Narrow_Fast", "VibrotactileOnly_D3_Narrow_Fast"],
        "VibrotactileOnly_Near_Slow": ["VibrotactileOnly_D1_Narrow_Slow", "VibrotactileOnly_D2_Narrow_Slow", "VibrotactileOnly_D3_Narrow_Slow"],

        "VibrotactileOnly_Far_Fast": ["VibrotactileOnly_D6_Narrow_Fast", "VibrotactileOnly_D5_Narrow_Fast", "VibrotactileOnly_D4_Narrow_Fast"],
        "VibrotactileOnly_Far_Slow": ["VibrotactileOnly_D6_Narrow_Slow", "VibrotactileOnly_D5_Narrow_Slow", "VibrotactileOnly_D4_Narrow_Slow"],
    }

    # Average tfr per defined groups
    tfr_grouped_single, tfr_grouped_average  = average_grouped_tfr(tfr_single_merged,groups)

### Save Time-Freq data

In [ ]:
if task=='Task1_PPS':
    folder_name = "TFR_interp_raw"
else:
    folder_name = "TFR_raw"

# Raw epoch TFR
file_path_raw = os.path.join(data_path, subjectID, task, "Output", folder_name)
filename = f"{subjectID}_{date}_{task}_raw"
save_paths = save_tfr_results(
    file_path=file_path_raw,
    file_name=filename,
    tfr_single_dict=tfr_single_raw,
    tfr_average_dict=tfr_average_raw
)


if task=='Task1_PPS':
    # Non grouped TFR but TFR interpolated for grouping
    file_path_merged = os.path.join(data_path, subjectID, task, "Output", "TFR_interp_grouped")
    filename = f"{subjectID}_{date}_{task}_interp_merged"
    save_paths = save_tfr_results(
        file_path=file_path_merged,
        file_name=filename,
        tfr_single_dict=tfr_single_merged,
        tfr_average_dict=tfr_average_merged
    )

    # Grouped TFR
    file_path_grouped = os.path.join(data_path, subjectID, task, "Output", "TFR_grouped")
    filename = f"{subjectID}_{date}_{task}_grouped"
    save_paths = save_tfr_results(
        file_path=file_path_grouped,
        file_name=filename,
        tfr_single_dict=tfr_grouped_single,
        tfr_average_dict=tfr_grouped_average,
        verbose=False
    )

    # Grouped categories used fot grouped TFR
    with open(os.path.join(file_path_grouped, "groups_dict.pkl"), "wb") as f:
        pickle.dump(groups, f)